# 📓 06 — Simulation: Policy Runner & RecordingRun policies inside the simulation, evaluate performance, apply domain randomization,and record trajectories to LeRobotDataset format.

## Running a Policy in SimulationThe `run_policy()` method orchestrates the full loop: `obs → policy(obs) → send_action → step()`.

In [ ]:
from strands_robots.simulation import create_simulationsim = create_simulation()sim.create_world(timestep=0.002)sim.add_robot("so100", data_config="so100", position=[0, 0, 0])sim.add_object("cube", shape="box", position=[0.3, 0, 0.05],               size=[0.03, 0.03, 0.03], color=[1, 0, 0, 1])print("World ready ✅")

In [ ]:
# Run MockPolicy in the simulationresult = sim.run_policy(    robot_name="so100",    policy_provider="mock",    instruction="pick up the red cube",    duration=2.0,          # seconds of sim time    record_video=None,     # set to "output.mp4" to record)print(result["content"][0]["text"])

## Domain RandomizationRandomize visual and physical properties for robust sim-to-real transfer.

In [ ]:
# Apply randomizationresult = sim.randomize(    randomize_colors=True,    randomize_lighting=True,    randomize_physics=True,    randomize_positions=True,    position_noise=0.02,    seed=42,)print(result["content"][0]["text"])

## Trajectory Recording to LeRobotDatasetRecord simulation rollouts directly to HuggingFace LeRobotDataset format (Parquet + video).```python# Start recordingsim.start_recording(    repo_id="user/my_sim_dataset",    task="pick up cube",    fps=30,    vcodec="libsvtav1",)# Run policy (frames are auto-recorded)sim.run_policy("so100", "mock", instruction="pick up cube", duration=5.0)# End episodesim.stop_recording()# Optionally push to HuggingFace Hub# sim.stop_recording(push_to_hub=True)```

## Video Recording```python# Record a policy rollout as videosim.run_policy(    "so100", "mock",    instruction="wave arm",    duration=3.0,    record_video="rollout.mp4",    video_fps=30,    video_camera="default",)# → rollout.mp4 saved```

## Eval Pattern```pythonimport numpy as np# Evaluate a policy across N episodes with randomizationsuccess_count = 0n_episodes = 10for ep in range(n_episodes):    sim.reset()    sim.randomize(seed=ep)    result = sim.run_policy("so100", "mock",                            instruction="pick up cube", duration=5.0)    # Check success condition (e.g., object height)    obs = sim.get_observation("so100")    # success = check_success(obs)    # if success: success_count += 1print(f"Success rate: {success_count}/{n_episodes}")```

In [ ]:
sim.destroy()print("Done ✅")